# Step-by-Step Input Preprocessing to Model Input (Single Example)

This notebook recreates the exact training-time preprocessing path in RINGER, from a raw `.pickle` molecule file to tensors consumed by `BertForDiffusion`.

It mirrors the style of `visualize_metrics_step_by_step.ipynb` with numbered sections and annotated outputs.

Pipeline we will reproduce:

1. Load config and choose one molecule
2. Inspect raw molecule and conformers
3. Compute internal coordinates
4. Build clean dataset sample (`Macrocycle...Dataset`)
5. Verify zero-centering + angular wrapping
6. Add diffusion noise (`NoisedDataset`)
7. Verify the noising equation numerically
8. Batch collation with `DataLoader`
9. Run model forward pass on that batch


In [1]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader
from transformers.models.bert import configuration_bert

import ringer
from ringer import data
from ringer.data import noised
from ringer.models import bert_for_diffusion
from ringer.utils import chem, internal_coords, utils

torch.manual_seed(0)
np.random.seed(0)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print(f"Repo config dir: {ringer.CONFIG_DIR}")
print(f"Repo data dir:   {ringer.DATA_DIR}")

/home/phamnh/miniforge3/envs/ringer/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo config dir: /mnt/HDD1/Codes/ringer/configs
Repo data dir:   /mnt/HDD1/Codes/ringer/data


## 1) Load a training config and select one example molecule

We use `configs/minimal_conditional.json` to keep runtime light while preserving the same preprocessing logic as full training.

**What to look for:**
- dataset type (`angles-sidechains`)
- diffusion settings (`timesteps`, schedule)
- model-relevant settings (hidden size, heads, etc.)

In [2]:
config_path = Path("configs/conditional_dummy.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

interesting = {
    k: cfg[k]
    for k in [
        "data_dir",
        "split_sizes",
        "internal_coordinates_definitions",
        "use_atom_features",
        "timesteps",
        "variance_schedule",
        "variance_scale",
        "time_encoding",
        "num_hidden_layers",
        "hidden_size",
        "num_heads",
        "decoder",
    ]
}
print("Selected config values:")
display(pd.Series(interesting))

data_dir = Path(cfg["data_dir"])
if not data_dir.exists():
    data_dir = ringer.DATA_DIR / data_dir

pickle_paths = sorted(data_dir.glob("*.pickle"))
example_path = pickle_paths[0]

print(f"Resolved data dir: {data_dir}")
print(f"Total molecules found: {len(pickle_paths)}")
print(f"Chosen example file: {example_path.name}")

Selected config values:


data_dir                                  cremp/dummy
split_sizes                           [0.9, 0.1, 0.0]
internal_coordinates_definitions    angles-sidechains
use_atom_features                                True
timesteps                                          20
variance_schedule                              cosine
variance_scale                                    1.0
time_encoding                        gaussian_fourier
num_hidden_layers                                  12
hidden_size                                       256
num_heads                                          12
decoder                                           mlp
dtype: object

Resolved data dir: /mnt/HDD1/Codes/ringer/data/cremp/dummy
Total molecules found: 10
Chosen example file: A.A.A.A.pickle


## 2) Inspect raw pickle object and molecule structure

A training file stores an RDKit molecule with many conformers (candidate 3D structures).

**What to look for:**
- number of atoms and conformers
- ring atom ordering used downstream (`n_to_c=True`)

In [3]:
with open(example_path, "rb") as f:
    obj = pickle.load(f)

mol = obj["rd_mol"] if isinstance(obj, dict) and "rd_mol" in obj else obj

print(f"Object type: {type(obj)}")
print(f"Molecule atoms: {mol.GetNumAtoms()}")
print(f"Molecule conformers: {mol.GetNumConformers()}")

macrocycle_idxs = chem.get_macrocycle_idxs(mol, n_to_c=True)
print(f"Macrocycle size: {len(macrocycle_idxs)} atoms")
print(f"First 15 macrocycle atom indices: {macrocycle_idxs[:15]}")

atom_table = pd.DataFrame(
    {
        "atom_idx": [a.GetIdx() for a in mol.GetAtoms()],
        "symbol": [a.GetSymbol() for a in mol.GetAtoms()],
        "atomic_num": [a.GetAtomicNum() for a in mol.GetAtoms()],
    }
)
display(atom_table.head(20))

Object type: <class 'dict'>
Molecule atoms: 40
Molecule conformers: 5
Macrocycle size: 12 atoms
First 15 macrocycle atom indices: [7, 5, 3, 2, 1, 18, 17, 15, 13, 12, 10, 8]


,atom_idx,symbol,atomic_num
0,0,C,6
1,1,C,6
2,2,N,7
3,3,C,6
4,4,O,8
5,5,C,6
6,6,C,6
7,7,N,7
8,8,C,6
9,9,O,8


## 3) Compute internal coordinates from this single file

This matches the featurization step used by dataset construction.

**What to look for:**
- per-feature matrices have shape: `[num_conformers x ring_length]`
- backbone metadata (`atom_labels`, `atom_ids`)
- side-chain internal coordinates (before flattening)

In [4]:
ic = internal_coords.get_macrocycle_distances_and_angles_from_file(
    example_path,
    include_side_chains=True,
)

print("Top-level keys:", list(ic.keys()))
print("atom_labels (first 12):", ic["atom_labels"][:12])
print("atom_ids    (first 12):", ic["atom_ids"][:12])

for name in ["distance", "angle", "dihedral"]:
    df = ic[name]
    print("\n{}: shape={}".format(name, df.shape))
    display(df.iloc[:2, :8])

side_chain_keys = list(ic["side_chains"].keys())
print(f"\nSide chains attached to {len(side_chain_keys)} backbone atoms")
if side_chain_keys:
    k = side_chain_keys[0]
    print(f"Example side-chain anchor atom: {k}")
    print("side-chain distance shape:", ic["side_chains"][k]["distance"].shape)
    print("side-chain angle shape:   ", ic["side_chains"][k]["angle"].shape)
    print("side-chain dihedral shape:", ic["side_chains"][k]["dihedral"].shape)

Top-level keys: ['atom_labels', 'atom_ids', 'distance', 'angle', 'dihedral', 'side_chains']
atom_labels (first 12): ['N', 'Calpha', 'CO', 'N', 'Calpha', 'CO', 'N', 'Calpha', 'CO', 'N', 'Calpha', 'CO']
atom_ids    (first 12): [0, 1, 2, 0, 1, 2, 0, 1, 2, 0, 1, 2]

distance: shape=(5, 12)


,7,5,3,2,1,18,17,15
conf_idx,,,,,,,,
0,1.460737,1.53829,1.344354,1.467749,1.534998,1.338974,1.460718,1.538304
1,1.459743,1.53780,1.351287,1.454628,1.535285,1.334150,1.468112,1.542827



angle: shape=(5, 12)


,7,5,3,2,1,18,17,15
conf_idx,,,,,,,,
0,2.123058,1.807548,1.987310,2.176574,1.840355,2.025294,2.123243,1.807633
1,2.123731,1.898565,1.992665,2.128711,1.815938,2.048193,2.191929,1.883690



dihedral: shape=(5, 12)


,7,5,3,2,1,18,17,15
conf_idx,,,,,,,,
0,-1.348506,1.476205,-2.783806,1.194501,-1.346701,2.832812,-1.348204,1.475668
1,-0.616469,-0.988351,2.646570,-1.859785,2.227772,-2.919037,1.178655,-1.239816



Side chains attached to 4 backbone atoms
Example side-chain anchor atom: 1
side-chain distance shape: (5, 1)
side-chain angle shape:    (5, 1)
side-chain dihedral shape: (5, 1)


In [15]:
ic["distance"]

,7,5,3,2,1,18,17,15,13,12,10,8
conf_idx,,,,,,,,,,,,
0,1.460737,1.538290,1.344354,1.467749,1.534998,1.338974,1.460718,1.538304,1.344334,1.467740,1.534999,1.338988
1,1.459743,1.537800,1.351287,1.454628,1.535285,1.334150,1.468112,1.542827,1.351117,1.454793,1.542598,1.334503
2,1.452072,1.539283,1.349118,1.459070,1.534026,1.348472,1.458028,1.537927,1.332184,1.464121,1.543671,1.339088
3,1.458996,1.533857,1.348124,1.458234,1.537975,1.332312,1.464013,1.543442,1.339221,1.452407,1.539229,1.349253
4,1.462656,1.538646,1.338673,1.466885,1.542210,1.343912,1.456586,1.537392,1.333829,1.459077,1.548351,1.351988


## 4) Build the clean dataset object used by training

Now we reproduce the exact dataset class selected by config (`angles-sidechains`).

**What to look for:**
- `feature_names` and angular flags
- global padding length (`pad`)
- training means used to zero-center each feature

In [6]:
clean_dset_class = data.DATASET_CLASSES[cfg["internal_coordinates_definitions"]]
print("Dataset class selected:", clean_dset_class.__name__)

clean_dset = clean_dset_class(
    data_dir=data_dir,
    split="train",
    split_sizes=cfg["split_sizes"],
    use_atom_features=cfg["use_atom_features"],
    fingerprint_radius=cfg.get("atom_feature_fingerprint_radius", 3),
    fingerprint_size=cfg.get("atom_feature_fingerprint_size", 32),
    num_conf=cfg["max_conf"],
    all_confs_in_test=True,
    zero_center=True,
    use_cache=True,
)

print(f"Dataset length (flattened conformers): {len(clean_dset)}")
print(f"pad (max sequence length): {clean_dset.pad}")
print("feature_names:", clean_dset.feature_names)
print("feature_is_angular:", clean_dset.feature_is_angular)

means_series = pd.Series(clean_dset.means_dict, name="training_mean")
display(means_series)

print("Atom feature table shape for selected example:")
example_abs = str(example_path.resolve())
if clean_dset.atom_features is not None and example_abs in clean_dset.atom_features:
    print(clean_dset.atom_features[example_abs]["atom_features"].shape)

Dataset class selected: MacrocycleAnglesWithSideChainsDataset
Dataset length (flattened conformers): 221
pad (max sequence length): 18
feature_names: ('angle', 'dihedral', 'sc_a0', 'sc_a1', 'sc_a2', 'sc_a3', 'sc_a4', 'sc_chi0', 'sc_chi1', 'sc_chi2', 'sc_chi3', 'sc_chi4')
feature_is_angular: (True, True, True, True, True, True, True, True, True, True, True, True)


/mnt/HDD1/Codes/ringer/ringer/utils/utils.py:109: RuntimeWarning: Mean of empty slice
  retval = np.arctan2(np.nanmean(sin_x, axis=axis), np.nanmean(cos_x, axis=axis))


angle       2.010828
dihedral   -3.031965
sc_a0       1.925896
sc_a1       1.954747
sc_a2       2.071725
sc_a3            NaN
sc_a4            NaN
sc_chi0    -3.041189
sc_chi1    -1.069868
sc_chi2    -2.937721
sc_chi3          NaN
sc_chi4          NaN
Name: training_mean, dtype: float64

Atom feature table shape for selected example:
(12, 178)


## 5) Extract one clean sample (`__getitem__`) and inspect model-ready fields

We pick one conformer from our chosen molecule and inspect the exact tensors produced by the clean dataset.

**What to look for:**
- `angles`: `[pad x num_features]`
- sequence masks: `attn_mask` and `feat_mask`
- conditioning fields: `atom_ids`, `atom_features`

In [7]:
matches = [
    i
    for i, (fname, _) in enumerate(clean_dset.global_conf_ids)
    if fname == example_abs
]

print(f"Conformers from chosen molecule present in TRAIN split: {len(matches)}")
dataset_idx = matches[0]
fname, conf_idx = clean_dset.global_conf_ids[dataset_idx]
print(f"Selected flattened dataset index: {dataset_idx}")
print(f"Underlying file: {Path(fname).name}, conformer index: {conf_idx}")

item_clean = clean_dset[dataset_idx]
for k, v in item_clean.items():
    if torch.is_tensor(v):
        print(f"{k:14s} shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"{k:14s} type={type(v)}")

seq_len = int(item_clean["lengths"].item())
print(f"\nUnpadded sequence length for this sample: {seq_len}")

angles_df = pd.DataFrame(
    item_clean["angles"][:seq_len].numpy(),
    columns=clean_dset.feature_names,
)
print("\nCentered+wrapped internal coords (first 5 rows):")
display(angles_df.head())

if "feat_mask" in item_clean:
    feat_mask_df = pd.DataFrame(
        item_clean["feat_mask"][:seq_len].numpy(),
        columns=clean_dset.feature_names,
    )
    print("Feature mask (1=present, 0=missing side-chain coordinate), first 5 rows:")
    display(feat_mask_df.head())

Conformers from chosen molecule present in TRAIN split: 5
Selected flattened dataset index: 30
Underlying file: A.A.A.A.pickle, conformer index: 0
angles         shape=(18, 12) dtype=torch.float32
attn_mask      shape=(18,) dtype=torch.float32
feat_mask      shape=(18, 12) dtype=torch.float32
position_ids   shape=(18,) dtype=torch.int64
atom_ids       shape=(18,) dtype=torch.int64
lengths        shape=() dtype=torch.int64
weights        shape=(12,) dtype=torch.float32
atom_features  shape=(18, 178) dtype=torch.float32

Unpadded sequence length for this sample: 12

Centered+wrapped internal coords (first 5 rows):


,angle,dihedral,sc_a0,sc_a1,sc_a2,sc_a3,sc_a4,sc_chi0,sc_chi1,sc_chi2,sc_chi3,sc_chi4
0,0.112231,1.683459,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,-0.203280,-1.775015,-0.002033,0.0,0.0,0.0,0.0,-0.407966,0.0,0.0,0.0,0.0
2,-0.023517,0.248160,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.165746,-2.056719,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,-0.170473,1.685265,0.051917,0.0,0.0,0.0,0.0,2.022208,0.0,0.0,0.0,0.0


Feature mask (1=present, 0=missing side-chain coordinate), first 5 rows:


,angle,dihedral,sc_a0,sc_a1,sc_a2,sc_a3,sc_a4,sc_chi0,sc_chi1,sc_chi2,sc_chi3,sc_chi4
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [8]:
item_clean

{'angles': tensor([[ 1.1223e-01,  1.6835e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [-2.0328e-01, -1.7750e+00, -2.0332e-03,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00, -4.0797e-01,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [-2.3517e-02,  2.4816e-01,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 1.6575e-01, -2.0567e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [-1.7047e-01,  1.6853e+00,  5.1917e-02,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00,  2.0222e+00,  0.0000e+00,  0.0000e+00,
           0.0000e+00,  0.0000e+00],
         [ 1.4467e-02, -4.1841e-01,  0.0000e+00,  0.000

## 6) Numerically verify zero-centering + wrapping against raw structure values

Here we reconstruct the dataset transform manually and check it matches `item_clean["angles"]`.

**What to look for:**
- max absolute difference should be ~0 (floating point tolerance)

In [9]:
structure = clean_dset.structures[example_abs]
raw = np.vstack([structure[name].loc[conf_idx].to_numpy() for name in clean_dset.feature_names]).T
raw_df = pd.DataFrame(raw, columns=clean_dset.feature_names)

manual = raw.copy()
manual = manual - clean_dset.means
manual[:, clean_dset.feature_is_angular] = utils.modulo_with_wrapped_range(
    manual[:, clean_dset.feature_is_angular],
    -np.pi,
    np.pi,
)
np.nan_to_num(manual, copy=False, nan=0.0)

from_dataset = item_clean["angles"][: raw.shape[0]].numpy()
max_abs_diff = np.max(np.abs(manual - from_dataset))
print(f"Max |manual - dataset| difference: {max_abs_diff:.6e}")

print("Raw internal coords (first 3 rows):")
display(raw_df.head(3))

print("After mean shift + wrapping + NaN->0 (first 3 rows):")
display(pd.DataFrame(manual, columns=clean_dset.feature_names).head(3))

Max |manual - dataset| difference: 1.173843e-07
Raw internal coords (first 3 rows):


,angle,dihedral,sc_a0,sc_a1,sc_a2,sc_a3,sc_a4,sc_chi0,sc_chi1,sc_chi2,sc_chi3,sc_chi4
0,2.123058,-1.348506,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.807548,1.476205,1.923863,NaN,NaN,NaN,NaN,2.834031,NaN,NaN,NaN,NaN
2,1.987310,-2.783806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


After mean shift + wrapping + NaN->0 (first 3 rows):


,angle,dihedral,sc_a0,sc_a1,sc_a2,sc_a3,sc_a4,sc_chi0,sc_chi1,sc_chi2,sc_chi3,sc_chi4
0,0.112231,1.683459,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,-0.203280,-1.775015,-0.002033,0.0,0.0,0.0,0.0,-0.407966,0.0,0.0,0.0,0.0
2,-0.023517,0.248160,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


## 7) Wrap clean dataset with diffusion noising (`NoisedDataset`)

This adds timestep-dependent noise and creates the supervised denoising target (`known_noise`).

**What to look for:**
- schedule tensors (`betas`, `sqrt_alphas_cumprod`, ...)
- sampled timestep `t` and noised tensor `corrupted`

In [10]:
noised_dset = noised.NoisedDataset(
    dset=clean_dset,
    dset_key="angles",
    timesteps=cfg["timesteps"],
    exhaustive_t=False,
    beta_schedule=cfg["variance_schedule"],
    nonangular_variance=1.0,
    angular_variance=cfg["variance_scale"],
    mask_noise=False,
    mask_noise_for_features=None,
)

print(f"timesteps: {noised_dset.timesteps}")
print(f"schedule:  {noised_dset.schedule}")

alpha_terms = noised_dset.alpha_beta_terms
sched_df = pd.DataFrame(
    {
        "beta": alpha_terms["betas"].numpy(),
        "sqrt_alpha_bar": alpha_terms["sqrt_alphas_cumprod"].numpy(),
        "sqrt_1m_alpha_bar": alpha_terms["sqrt_one_minus_alphas_cumprod"].numpy(),
    }
)
display(sched_df.head(10))

timesteps: 20
schedule:  cosine


,beta,sqrt_alpha_bar,sqrt_1m_alpha_bar
0,0.007993,0.995996,0.089402
1,0.020075,0.985948,0.167055
2,0.032254,0.969917,0.243436
3,0.044681,0.948001,0.318267
4,0.057520,0.920333,0.391137
5,0.070957,0.887080,0.461616
6,0.085210,0.848444,0.529285
7,0.100547,0.804660,0.593735
8,0.117304,0.755994,0.654579
9,0.135922,0.702740,0.711447


## 8) Verify the noising equation for one fixed timestep

Forward diffusion equation in code:

\[x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon\]

with angular dimensions wrapped back to `[-pi, pi]`.

**What to look for:**
- `max |reconstructed - corrupted|` should be ~0

In [11]:
t_demo = min(7, cfg["timesteps"] - 1)
item_noised = noised_dset.__getitem__(dataset_idx, use_t_val=t_demo)

for k in ["corrupted", "known_noise", "t", "sqrt_alphas_cumprod_t", "sqrt_one_minus_alphas_cumprod_t"]:
    v = item_noised[k]
    if torch.is_tensor(v):
        print(f"{k:28s} shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"{k:28s} value={v}")

x0 = item_clean["angles"].clone()
eps = item_noised["known_noise"].clone()
x_t = item_noised["corrupted"].clone()

sqrt_ab = item_noised["sqrt_alphas_cumprod_t"]
sqrt_1mab = item_noised["sqrt_one_minus_alphas_cumprod_t"]
x_t_manual = sqrt_ab * x0 + sqrt_1mab * eps
x_t_manual[:, clean_dset.feature_is_angular] = utils.modulo_with_wrapped_range(
    x_t_manual[:, clean_dset.feature_is_angular],
    -np.pi,
    np.pi,
)

max_abs_diff = torch.max(torch.abs(x_t_manual - x_t)).item()
print(f"\nMax |manual noising - dataset corrupted| difference: {max_abs_diff:.6e}")

demo_df = pd.DataFrame(
    {
        "x0(f0)": x0[:8, 0].numpy(),
        "eps(f0)": eps[:8, 0].numpy(),
        "x_t dataset(f0)": x_t[:8, 0].numpy(),
        "x_t manual(f0)": x_t_manual[:8, 0].numpy(),
    }
)
display(demo_df)

corrupted                    shape=(18, 12) dtype=torch.float32
known_noise                  shape=(18, 12) dtype=torch.float32
t                            shape=(1,) dtype=torch.int64
sqrt_alphas_cumprod_t        shape=() dtype=torch.float32
sqrt_one_minus_alphas_cumprod_t shape=() dtype=torch.float32

Max |manual noising - dataset corrupted| difference: 0.000000e+00


,x0(f0),eps(f0),x_t dataset(f0),x_t manual(f0)
0,0.112231,1.540996,1.005251,1.005251
1,-0.203280,-0.293429,-0.337790,-0.337790
2,-0.023517,-2.178789,-1.312548,-1.312548
3,0.165746,0.568431,0.470867,0.470867
4,-0.170473,-1.084522,-0.781092,-0.781092
5,0.014467,-1.398595,-0.818755,-0.818755
6,0.112415,0.403347,0.329937,0.329937
7,-0.203195,0.838026,0.334063,0.334063


## 9) Build a mini-batch with `DataLoader`

This shows exactly what the Lightning `training_step` receives: batched tensors for corrupted input, timestep, masks, and conditioning features.

**What to look for:**
- batch dimensions: `[B, pad, num_features]` for `corrupted` and `known_noise`
- per-sample timestep tensor shape `[B, 1]`

In [12]:
loader = DataLoader(noised_dset, batch_size=4, shuffle=False, num_workers=0)
batch = next(iter(loader))

print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"{k:28s} shape={tuple(v.shape)} dtype={v.dtype}")

print("\nFirst sample sequence length from attn_mask sum:", int(batch["attn_mask"][0].sum().item()))
print("First sample timestep t:", int(batch["t"][0].item()))

Batch keys: ['angles', 'attn_mask', 'feat_mask', 'position_ids', 'atom_ids', 'lengths', 'weights', 'atom_features', 'corrupted', 't', 'known_noise', 'sqrt_alphas_cumprod_t', 'sqrt_one_minus_alphas_cumprod_t']
angles                       shape=(4, 18, 12) dtype=torch.float32
attn_mask                    shape=(4, 18) dtype=torch.float32
feat_mask                    shape=(4, 18, 12) dtype=torch.float32
position_ids                 shape=(4, 18) dtype=torch.int64
atom_ids                     shape=(4, 18) dtype=torch.int64
lengths                      shape=(4,) dtype=torch.int64
weights                      shape=(4, 12) dtype=torch.float32
atom_features                shape=(4, 18, 178) dtype=torch.float32
corrupted                    shape=(4, 18, 12) dtype=torch.float32
t                            shape=(4, 1) dtype=torch.int64
known_noise                  shape=(4, 18, 12) dtype=torch.float32
sqrt_alphas_cumprod_t        shape=(4,) dtype=torch.float32
sqrt_one_minus_alphas_cumprod

## 10) Instantiate model and run one forward pass

Now we feed the preprocessed batch into `BertForDiffusion` exactly as training does.

**What to look for:**
- output shape should match `known_noise` shape
- each output channel corresponds to one internal-coordinate feature

In [13]:
sample_item = noised_dset[0]
model_n_inputs = sample_item["corrupted"].shape[-1]
atom_feature_size = sample_item["atom_features"].shape[-1] if "atom_features" in sample_item else None

bert_cfg = configuration_bert.BertConfig(
    max_position_embeddings=noised_dset.pad,
    num_attention_heads=cfg["num_heads"],
    hidden_size=cfg["hidden_size"],
    intermediate_size=cfg["intermediate_size"],
    num_hidden_layers=cfg["num_hidden_layers"],
    position_embedding_type=cfg["position_embedding_type"],
    hidden_dropout_prob=cfg["dropout_p"],
    attention_probs_dropout_prob=cfg["dropout_p"],
    use_cache=False,
)

model = bert_for_diffusion.BertForDiffusion(
    config=bert_cfg,
    ft_is_angular=noised_dset.feature_is_angular,
    ft_names=noised_dset.feature_names,
    time_encoding=cfg["time_encoding"],
    decoder=cfg["decoder"],
    atom_feature_size=atom_feature_size,
    atom_feature_embed_size=cfg["atom_feature_embed_size"],
    lr=cfg["lr"],
    loss=cfg["loss"],
    use_feat_mask=cfg.get("use_feat_mask", True),
    l2=cfg["l2_norm"],
    l1=cfg["l1_norm"],
    circle_reg=cfg["circle_reg"],
    epochs=cfg["max_epochs"],
    warmup_epochs=cfg["warmup_epochs"],
    lr_scheduler=cfg["lr_scheduler"],
)

model.eval()
with torch.no_grad():
    pred_noise = model(
        inputs=batch["corrupted"],
        timestep=batch["t"],
        attention_mask=batch["attn_mask"],
        position_ids=batch["position_ids"],
        atom_ids=batch["atom_ids"],
        atom_features=batch.get("atom_features"),
    )

print("predicted_noise shape:", tuple(pred_noise.shape))
print("known_noise     shape:", tuple(batch["known_noise"].shape))
print("Shapes match:", tuple(pred_noise.shape) == tuple(batch["known_noise"].shape))

with torch.no_grad():
    loss_terms = model._get_loss_terms(batch)

loss_df = pd.DataFrame(
    {
        "feature": list(noised_dset.feature_names),
        "loss_term": loss_terms.cpu().numpy(),
    }
)
display(loss_df)

Using time embedding: GaussianFourierProjection()
Using loss: [functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, beta=0.3141592653589793), functools.partial(<function radian_smooth_l1_loss at 0x7ba615344550>, bet

predicted_noise shape: (4, 18, 12)
known_noise     shape: (4, 18, 12)
Shapes match: True


,feature,loss_term
0,angle,0.774233
1,dihedral,0.672766
2,sc_a0,0.761314
3,sc_a1,0.664344
4,sc_a2,0.691059
5,sc_a3,0.677362
6,sc_a4,0.841525
7,sc_chi0,0.643050
8,sc_chi1,0.859148
9,sc_chi2,0.555257


## 11) Final tensor map (raw file -> model input)

Use this as a compact reference:

- raw file (`.pickle`) -> RDKit mol with many conformers
- internal coordinate extraction -> `distance/angle/dihedral` tables
- clean dataset item -> `angles`, masks, ids, optional atom features
- noised dataset item -> `corrupted`, `known_noise`, `t`, schedule scalars
- batch collation -> add leading batch dimension
- model forward input -> `(corrupted, t, attention_mask, position_ids, atom_ids, atom_features)`
- model output -> predicted noise with same shape as `known_noise`

You can now swap `minimal_conditional.json` for `conditional.json` to inspect the full CREMP setup with the same steps.